# Fase 1 – Load Data

Pada tahap ini dilakukan koneksi ke Google Drive, pemuatan dataset ke dalam notebook, serta import library yang diperlukan untuk proses eksplorasi data. Selain itu, kolom bertipe waktu dikonversi ke format `datetime` agar dapat digunakan untuk analisis berbasis waktu pada tahap selanjutnya.

In [ ]:
# ============================================================
# FASE 1 — LOAD DATA
# ============================================================

from pathlib import Path

# Root project
BASE = Path("..").resolve()

DATA_DIR = BASE / "data"
OUTPUT_DIR = BASE / "output"
MODEL_DIR = BASE / "models"
ASSET_DIR = BASE / "assets"

OUTPUT_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)
ASSET_DIR.mkdir(exist_ok=True)

DATA_PATH = DATA_DIR / "AI_Engineer_dataset.parquet"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_parquet(DATA_PATH)

df["arrival_time"] = pd.to_datetime(df["arrival_time"])
df["from_arrival_time_str"] = pd.to_datetime(df["from_arrival_time_str"])

print(df.shape)
df.head()

Mounted at /content/drive


# Fase 2 – Data Preparation & Cleaning

Tahap ini bertujuan menyiapkan data untuk proses analisis dengan merekonstruksi perjalanan aktual (`real_trip_id`) dan menghapus data duplikat akibat pencatatan berulang pada segmen yang sama. Hasilnya adalah dataset yang lebih konsisten dan siap digunakan pada tahap eksplorasi maupun pemodelan.

In [ ]:
# ============================================================
# FASE 2 — DATA CLEANING & PREPARATION
# ============================================================

# Rekonstruksi real_trip_id (menangkap loop berulang dalam 1 no_do)
df = df.sort_values(['no_do', 'arrival_time']).reset_index(drop=True)
df['prev_seq'] = df.groupby('no_do')['stop_sequence'].shift(1)
df['seq_reset'] = (df['stop_sequence'] < (df['prev_seq'] - 2)).fillna(False)
df['trip_instance'] = df.groupby('no_do')['seq_reset'].cumsum()
df['real_trip_id'] = df['no_do'].astype(str) + '_' + df['trip_instance'].astype(str)

# Dwell dedup (sisa duplikat stop_sequence dalam 1 real_trip_id, ambil baris terakhir)
df = df.sort_values(['real_trip_id', 'stop_sequence', 'arrival_time'])
df['is_dwell_duplicate'] = df.duplicated(['real_trip_id', 'stop_sequence'], keep='last')
df_clean = df[~df['is_dwell_duplicate']].copy()

print(f"Shape sebelum dedup : {df.shape}")
print(f"Shape setelah dedup : {df_clean.shape}")
print(f"Baris yang dihapus  : {len(df)-len(df_clean):,}")

print()

print(f"Jumlah no_do        : {df['no_do'].nunique():,}")
print(f"Jumlah real_trip_id : {df_clean['real_trip_id'].nunique():,}")


Shape awal: (351103, 15) -> setelah dedup: (350919, 15)
no_do unik: 17857 -> real_trip_id unik: 18344


## Fase 3 — Data Quality Flagging: Gap, Anomali & Posisi Segmen

Membangun kolom-kolom wajib yang menandai kualitas data per baris, di atas `real_trip_id` yang sudah bersih dari Fase 2:

- **`is_gap_suspected`** — menandai lompatan `stop_sequence` dalam satu trip (indikasi baris/halte hilang di tengah trip). **`incomplete_start_suspected`** menandai trip yang tidak mulai dari halte pertama.
- **`deviation_ratio`** (`traveling_time_sec / average_time_sec`) sebagai indikator anomali relatif terhadap baseline historis, dengan **dua tingkat flag**:
  - `anomaly_suspected` — rasio > 1.5, menangkap macet/insiden wajar
  - `data_error_suspected` — nilai absolut `traveling_time_sec` di atas persentil 99.9, karena `average_time_sec` ikut terkontaminasi pada nilai ekstrem (korelasi log-log ≈0.9) sehingga rasio saja tidak cukup menangkap data error
- **`is_first_segment`** — flag domain insight: seluruh kasus `data_error_suspected` terkonsentrasi di `stop_sequence == 1` (kemungkinan waktu dwell/idle di halte awal tercampur ke `traveling_time_sec`)
- **`exclude_from_target`** — flag baris yang sebaiknya dikeluarkan dari target (`y`) saat training, tanpa menghapusnya dari dataframe

Output disimpan sebagai checkpoint (`cleaned_segments_checkpoint.parquet`) untuk dipakai di Fase 4.

In [4]:
# ============================================================
# FASE 3 — Data Quality Flagging — Gap, Anomali & Posisi Segmen
# ============================================================

df_clean = df_clean.sort_values(['real_trip_id', 'stop_sequence'])

# is_gap_suspected
df_clean['seq_diff'] = df_clean.groupby('real_trip_id')['stop_sequence'].diff()
df_clean['is_gap_suspected'] = df_clean['seq_diff'].gt(1).fillna(False)

first_seq = df_clean.groupby('real_trip_id')['stop_sequence'].transform('min')
df_clean['incomplete_start_suspected'] = (df_clean['stop_sequence'] == first_seq) & (first_seq > 1)

In [5]:
# deviation_ratio + dua tingkat anomaly flag
df_clean['deviation_ratio'] = df_clean['traveling_time_sec'] / df_clean['average_time_sec']
df_clean['log_deviation_ratio'] = np.log1p(df_clean['deviation_ratio'])

ANOMALY_RATIO_THRESHOLD = 1.5
DATA_ERROR_TIME_THRESHOLD = df_clean['traveling_time_sec'].quantile(0.999)

df_clean['anomaly_suspected'] = df_clean['deviation_ratio'] > ANOMALY_RATIO_THRESHOLD
df_clean['data_error_suspected'] = df_clean['traveling_time_sec'] > DATA_ERROR_TIME_THRESHOLD

In [ ]:
# fitur posisi segmen (domain insight: stop_sequence=1 punya pola dwell berbeda)
df_clean['is_first_segment'] = (df_clean['stop_sequence'] == first_seq)

# flag exclude untuk target training (bukan drop dari dataframe)
df_clean['exclude_from_target'] = df_clean['data_error_suspected']

print('is_gap_suspected:', df_clean['is_gap_suspected'].mean())
print('incomplete_start_suspected (per trip):',
      df_clean.groupby('real_trip_id')['incomplete_start_suspected'].max().mean())
print('anomaly_suspected:', df_clean['anomaly_suspected'].mean())
print('data_error_suspected:', df_clean['data_error_suspected'].mean())

# Simpan checkpoint
checkpoint_path = OUTPUT_DIR / "cleaned_segments_checkpoint.parquet"

df_clean.to_parquet(checkpoint_path)

print(f"Checkpoint saved to: {checkpoint_path}")
print(f"Shape checkpoint    : {df_clean.shape}")

is_gap_suspected: 0.03557231155907774
incomplete_start_suspected (per trip): 0.06323593545573485
anomaly_suspected: 0.01154682419589706
data_error_suspected: 0.0010002308224974993
Checkpoint saved: (350919, 24)


## Fase 4 — Feature Engineering

Membangun fitur tambahan dari kolom waktu dan relasi `traveling_time_sec` vs `average_time_sec`, di atas data bersih hasil Fase 2-3:

- **Fitur waktu**: `hour`, `day_of_week`, `is_weekend`, `is_peak_hour`, `time_bucket` — dari `arrival_time`, untuk menangkap variasi travel time akibat jam sibuk vs sepi dan hari kerja vs weekend.
- **Fitur posisi dalam trip**: `trip_progress` (posisi relatif segmen dalam trip) — relevan untuk pola LSTM sekuensial.
- **Fitur sekuensial (lag)**: `prev_deviation_ratio`, `prev_traveling_time_sec`, `cum_avg_deviation_in_trip` — menangkap delay yang menjalar antar segmen dalam satu trip yang sama.
- **Historical stats dengan hierarchical backoff** (`segment_time_estimate`): mengatasi sparsity pada kombinasi segmen-jam tertentu dengan mundur ke level lebih kasar (segmen+jam+weekend → segmen+jam → segmen saja) kalau jumlah sampel di level granular kurang dari `MIN_SAMPLES_LEVEL1`.
- **Fitur performa kendaraan**: `bus_avg_deviation`, `bus_trip_count` — historis rata-rata deviasi per `bus_body_no`.
- **Encoding kategorikal** (`*_enc`) untuk kolom seperti `segment_id`, `route_code`, `trip_id`, `bus_body_no`, `time_bucket` — dipakai oleh XGBoost (CatBoost nanti bisa native tanpa encoding ini).

> **Catatan leakage**: statistik di atas (`segment_time_estimate`, `bus_avg_deviation`) di sini dihitung dari **seluruh data** untuk keperluan eksplorasi. Saat training di Fase 5, statistik ini wajib dihitung ulang hanya dari train set untuk menghindari data leakage ke test set.

Output disimpan sebagai `featured_segments.parquet` untuk dipakai di Fase 5.

In [ ]:
# ============================================================
# FASE 4 — FEATURE ENGINEERING
# ============================================================

df_clean = df_clean.sort_values(['real_trip_id', 'stop_sequence']).reset_index(drop=True)

# --- 4.1 Fitur waktu (dari arrival_time) ---
df_clean['hour'] = df_clean['arrival_time'].dt.hour
df_clean['day_of_week'] = df_clean['arrival_time'].dt.dayofweek  # 0=Senin, 6=Minggu
df_clean['is_weekend'] = df_clean['day_of_week'].isin([5, 6])
df_clean['is_peak_hour'] = df_clean['hour'].isin([6, 7, 8, 16, 17, 18, 19])

def time_bucket(h):
    if 5 <= h < 10: return 'morning_peak'
    if 10 <= h < 15: return 'midday'
    if 15 <= h < 20: return 'evening_peak'
    if 20 <= h < 24: return 'night'
    return 'early_dawn'
df_clean['time_bucket'] = df_clean['hour'].apply(time_bucket)

# --- 4.2 Fitur posisi dalam trip ---
trip_len = df_clean.groupby('real_trip_id')['stop_sequence'].transform('max')
df_clean['trip_progress'] = df_clean['stop_sequence'] / trip_len

# --- 4.3 Fitur sekuensial (lag dalam trip yang sama) ---
df_clean['prev_deviation_ratio'] = df_clean.groupby('real_trip_id')['deviation_ratio'].shift(1)
df_clean['prev_traveling_time_sec'] = df_clean.groupby('real_trip_id')['traveling_time_sec'].shift(1)
df_clean['prev_deviation_ratio'] = df_clean['prev_deviation_ratio'].fillna(1.0)

df_clean['cum_avg_deviation_in_trip'] = (
    df_clean.groupby('real_trip_id')['deviation_ratio']
    .transform(lambda x: x.shift(1).expanding().mean())
    .fillna(1.0)
)

# --- 4.4 Historical stats per segmen dengan hierarchical backoff (atasi sparsity) ---
# CATATAN: ini versi GLOBAL untuk eksplorasi/EDA. Saat training di Fase 5,
# statistik ini WAJIB dihitung ulang hanya dari train set untuk hindari leakage.

MIN_SAMPLES_LEVEL1 = 20

stats_l1 = df_clean.groupby(['segment_id', 'hour', 'is_weekend'])['traveling_time_sec'].agg(
    seg_hour_weekend_mean='mean', seg_hour_weekend_count='count'
).reset_index()

stats_l2 = df_clean.groupby(['segment_id', 'hour'])['traveling_time_sec'].agg(
    seg_hour_mean='mean', seg_hour_count='count'
).reset_index()

stats_l3 = df_clean.groupby(['segment_id'])['traveling_time_sec'].agg(
    seg_mean='mean', seg_count='count'
).reset_index()

df_clean = df_clean.merge(stats_l1, on=['segment_id', 'hour', 'is_weekend'], how='left')
df_clean = df_clean.merge(stats_l2, on=['segment_id', 'hour'], how='left')
df_clean = df_clean.merge(stats_l3, on=['segment_id'], how='left')

df_clean['segment_time_estimate'] = np.where(
    df_clean['seg_hour_weekend_count'] >= MIN_SAMPLES_LEVEL1,
    df_clean['seg_hour_weekend_mean'],
    np.where(
        df_clean['seg_hour_count'] >= MIN_SAMPLES_LEVEL1,
        df_clean['seg_hour_mean'],
        df_clean['seg_mean']
    )
)

# --- 4.5 Fitur performa kendaraan (bus_body_no historis) ---
bus_stats = df_clean.groupby('bus_body_no')['deviation_ratio'].agg(
    bus_avg_deviation='mean', bus_trip_count='count'
).reset_index()
df_clean = df_clean.merge(bus_stats, on='bus_body_no', how='left')

# --- 4.6 Encoding kategorikal untuk XGBoost ---
from sklearn.preprocessing import LabelEncoder

cat_cols = ['segment_id', 'route_code', 'trip_id', 'bus_body_no', 'time_bucket']
label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df_clean[f'{col}_enc'] = le.fit_transform(df_clean[col].astype(str))
    label_encoders[col] = le

import joblib

joblib.dump(
    label_encoders,
    MODEL_DIR / "label_encoders.pkl"
)

print("Label encoders saved.")

print(f"Jumlah kolom akhir : {len(df_clean.columns)}")
print(f"Shape akhir        : {df_clean.shape}")
print('\nKolom baru hasil Fase 4:')
print([c for c in df_clean.columns if c not in ['bus_body_no','segment_id','route_code','trip_id',
      'stop_sequence','traveling_time_sec','from_arrival_time_str','arrival_time','average_time_sec','no_do']])

featured_path = OUTPUT_DIR / "featured_segments.parquet"

df_clean.to_parquet(featured_path)

print(f"\nFeature dataset saved to: {featured_path}")

Jumlah kolom akhir: 47
Shape akhir: (350919, 47)

Kolom baru hasil Fase 4:
['prev_seq', 'seq_reset', 'trip_instance', 'real_trip_id', 'is_dwell_duplicate', 'seq_diff', 'is_gap_suspected', 'incomplete_start_suspected', 'deviation_ratio', 'log_deviation_ratio', 'anomaly_suspected', 'data_error_suspected', 'is_first_segment', 'exclude_from_target', 'hour', 'day_of_week', 'is_weekend', 'is_peak_hour', 'time_bucket', 'trip_progress', 'prev_deviation_ratio', 'prev_traveling_time_sec', 'cum_avg_deviation_in_trip', 'seg_hour_weekend_mean', 'seg_hour_weekend_count', 'seg_hour_mean', 'seg_hour_count', 'seg_mean', 'seg_count', 'segment_time_estimate', 'bus_avg_deviation', 'bus_trip_count', 'segment_id_enc', 'route_code_enc', 'trip_id_enc', 'bus_body_no_enc', 'time_bucket_enc']

Saved to featured_segments.parquet


## Fase 5 — Modeling: Ensemble LSTM + XGBoost

Membangun model prediksi travel time dengan dua pendekatan yang saling melengkapi:
- **XGBoost**: menangkap pola tabular (rute, jam, fitur historis) — setiap baris segmen diperlakukan independen.
- **LSTM**: menangkap pola sekuensial antar segmen dalam satu trip (delay yang menjalar dari segmen ke segmen).

**Split dilakukan berbasis waktu DAN per-trip** (bukan random per baris): seluruh `real_trip_id` masuk ke train atau test secara utuh, dan trip yang lebih baru dipisah ke test set. Ini mencegah dua leakage sekaligus: (1) leakage temporal dari trip masa depan masuk ke train, dan (2) leakage sekuensial dari sebagian baris satu trip ada di train sementara sisanya di test (krusial untuk LSTM).

Statistik historis (`segment_time_estimate`, `bus_avg_deviation`) yang dihitung global di Fase 4 **dihitung ulang khusus dari train set** di sini untuk menghindari target leakage.

> **Model pembanding (opsional)**: CatBoost dilatih dengan fitur yang sama seperti XGBoost, namun memanfaatkan **native categorical handling** (tanpa label-encoding manual) pada kolom seperti `segment_id`, `route_code`, `bus_body_no`. Ini relevan karena CatBoost punya regularisasi ordered boosting yang lebih tahan overfitting pada kombinasi kategorikal jarang — langsung menjawab masalah sparsity segmen-jam yang ditemukan di Fase 2. Hasil dibandingkan dengan XGBoost untuk melihat apakah native categorical encoding memberi keuntungan nyata.

**Output**: prediksi tiap model + beberapa skema ensemble (weighted average) dievaluasi lebih lanjut di Fase 6.

In [ ]:
# ============================================================
# 5.1 — TRAIN/TEST SPLIT (TIME-BASED, PER TRIP)
# ============================================================
from sklearn.preprocessing import StandardScaler

feature_path = OUTPUT_DIR / "featured_segments.parquet"
df_feat = pd.read_parquet(feature_path)

print(f"Loaded feature dataset: {df_feat.shape}")

trip_start_time = df_feat.groupby('real_trip_id')['arrival_time'].min().sort_values()
cutoff = trip_start_time.quantile(0.8)  # 80% trip lebih awal -> train, 20% terakhir -> test

train_trip_ids = trip_start_time[trip_start_time <= cutoff].index
test_trip_ids = trip_start_time[trip_start_time > cutoff].index

print('Cutoff waktu:', cutoff)
print('Jumlah trip train:', len(train_trip_ids), '| test:', len(test_trip_ids))

train_df = df_feat[df_feat['real_trip_id'].isin(train_trip_ids)].copy()
test_df = df_feat[df_feat['real_trip_id'].isin(test_trip_ids)].copy()

print('Baris train:', train_df.shape[0], '| Baris test:', test_df.shape[0])

Cutoff waktu: 2026-02-23 15:14:03.600000
Jumlah trip train: 14675 | test: 3669
Baris train: 281818 | Baris test: 69101


In [29]:
# ============================================================
# 5.2 — RE-KALKULASI STATISTIK HANYA DARI TRAIN SET (HINDARI LEAKAGE)
# ============================================================
MIN_SAMPLES_LEVEL1 = 20

leak_cols = ['seg_hour_weekend_mean','seg_hour_weekend_count','seg_hour_mean',
             'seg_hour_count','seg_mean','seg_count','segment_time_estimate',
             'bus_avg_deviation','bus_trip_count']
train_df = train_df.drop(columns=leak_cols)
test_df = test_df.drop(columns=leak_cols)

stats_l1 = train_df.groupby(['segment_id','hour','is_weekend'])['traveling_time_sec'].agg(
    seg_hour_weekend_mean='mean', seg_hour_weekend_count='count').reset_index()
stats_l2 = train_df.groupby(['segment_id','hour'])['traveling_time_sec'].agg(
    seg_hour_mean='mean', seg_hour_count='count').reset_index()
stats_l3 = train_df.groupby(['segment_id'])['traveling_time_sec'].agg(
    seg_mean='mean', seg_count='count').reset_index()
bus_stats = train_df.groupby('bus_body_no')['deviation_ratio'].agg(
    bus_avg_deviation='mean', bus_trip_count='count').reset_index()

global_mean_train = train_df['traveling_time_sec'].mean()
global_bus_dev_train = train_df['deviation_ratio'].mean()

def apply_stats(d):
    d = d.merge(stats_l1, on=['segment_id','hour','is_weekend'], how='left')
    d = d.merge(stats_l2, on=['segment_id','hour'], how='left')
    d = d.merge(stats_l3, on=['segment_id'], how='left')
    d = d.merge(bus_stats, on='bus_body_no', how='left')

    d['segment_time_estimate'] = np.where(
        d['seg_hour_weekend_count'] >= MIN_SAMPLES_LEVEL1, d['seg_hour_weekend_mean'],
        np.where(d['seg_hour_count'] >= MIN_SAMPLES_LEVEL1, d['seg_hour_mean'], d['seg_mean'])
    )
    d['segment_time_estimate'] = d['segment_time_estimate'].fillna(global_mean_train)
    d['bus_avg_deviation'] = d['bus_avg_deviation'].fillna(global_bus_dev_train)
    d['bus_trip_count'] = d['bus_trip_count'].fillna(0)
    return d

train_df = apply_stats(train_df)
test_df = apply_stats(test_df)

print('Unseen segment di test:', test_df['segment_time_estimate'].isna().sum())
print('Unseen bus di test (sebelum fillna):', test_df['bus_avg_deviation'].isna().sum())

Unseen segment di test: 0
Unseen bus di test (sebelum fillna): 0


In [30]:
# ============================================================
# 5.3 — PERSIAPAN DATA TABULAR (TARGET & SPLIT X/y)
# ============================================================
from sklearn.metrics import mean_absolute_error

FEATURES_ENC = [
    'stop_sequence','hour','day_of_week','trip_progress',
    'prev_deviation_ratio','prev_traveling_time_sec','cum_avg_deviation_in_trip',
    'average_time_sec','segment_time_estimate','bus_avg_deviation','bus_trip_count',
    'is_weekend','is_peak_hour','is_first_segment','is_gap_suspected','incomplete_start_suspected',
    'segment_id_enc','route_code_enc','trip_id_enc','bus_body_no_enc','time_bucket_enc'
]
CAT_FEATURES_RAW = ['segment_id', 'route_code', 'trip_id', 'bus_body_no', 'time_bucket']
FEATURES_RAW = [
    'stop_sequence','hour','day_of_week','trip_progress',
    'prev_deviation_ratio','prev_traveling_time_sec','cum_avg_deviation_in_trip',
    'average_time_sec','segment_time_estimate','bus_avg_deviation','bus_trip_count',
    'is_weekend','is_peak_hour','is_first_segment','is_gap_suspected','incomplete_start_suspected'
] + CAT_FEATURES_RAW

TARGET = 'traveling_time_sec'

# Buang baris data_error_suspected dari TARGET training (bukan dari dataframe keseluruhan)
train_xgb = train_df[~train_df['exclude_from_target']].copy()
test_xgb = test_df[~test_df['exclude_from_target']].copy()

# --- versi encoded (untuk XGBoost) ---
X_train = train_xgb[FEATURES_ENC].astype(float)
X_test = test_xgb[FEATURES_ENC].astype(float)

# --- versi raw kategorikal (untuk CatBoost, native handling) ---
X_train_cb = train_xgb[FEATURES_RAW].copy()
X_test_cb = test_xgb[FEATURES_RAW].copy()
for c in CAT_FEATURES_RAW:
    X_train_cb[c] = X_train_cb[c].astype(str)
    X_test_cb[c] = X_test_cb[c].astype(str)

# --- target sama untuk semua model tabular ---
y_train = np.log1p(train_xgb[TARGET])
y_test = np.log1p(test_xgb[TARGET])

print('X_train (XGB):', X_train.shape, '| X_train (CatBoost):', X_train_cb.shape)
print('X_test (XGB):', X_test.shape, '| X_test (CatBoost):', X_test_cb.shape)

X_train (XGB): (281554, 21) | X_train (CatBoost): (281554, 21)
X_test (XGB): (69014, 21) | X_test (CatBoost): (69014, 21)


In [ ]:
# ============================================================
# 5.4 — TRAINING XGBOOST
# ============================================================
import xgboost as xgb

xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    max_depth=7,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='reg:squarederror',
    early_stopping_rounds=30,
    eval_metric='mae',
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50
)

# ============================================================
# SAVE XGBOOST MODEL
# ============================================================
import joblib

joblib.dump(
    xgb_model,
    MODEL_DIR / "xgb_model.pkl"
)

print(f"XGBoost model saved to: {MODEL_DIR / 'xgb_model.pkl'}")

print("✓ XGBoost model saved.")

xgb_pred_log = xgb_model.predict(X_test)
xgb_pred_sec = np.expm1(xgb_pred_log)

xgb_mae = mean_absolute_error(
    test_xgb[TARGET],
    xgb_pred_sec
)

print(f"XGBoost MAE : {xgb_mae:.2f} detik")

imp = pd.Series(xgb_model.feature_importances_, index=FEATURES_ENC).sort_values(ascending=False)
print(imp.head(10))

[0]	validation_0-mae:0.59344
[50]	validation_0-mae:0.15886
[100]	validation_0-mae:0.14387
[150]	validation_0-mae:0.14256
[200]	validation_0-mae:0.14204
[250]	validation_0-mae:0.14182
[300]	validation_0-mae:0.14163
[350]	validation_0-mae:0.14157
[368]	validation_0-mae:0.14157
✓ XGBoost model saved.
XGBoost MAE (detik): 105.35102336349556
average_time_sec              0.572319
segment_time_estimate         0.144064
incomplete_start_suspected    0.037535
stop_sequence                 0.028780
segment_id_enc                0.028742
is_peak_hour                  0.026110
prev_traveling_time_sec       0.023186
hour                          0.022196
bus_body_no_enc               0.017071
time_bucket_enc               0.016240
dtype: float32


In [ ]:
# ============================================================
# 5.4b — TRAINING CATBOOST (OPSIONAL — MODEL PEMBANDING)
# ============================================================
from catboost import CatBoostRegressor

cat_model = CatBoostRegressor(
    iterations=500,
    depth=7,
    learning_rate=0.05,
    cat_features=CAT_FEATURES_RAW,
    loss_function='MAE',          # robust terhadap skew, beda dgn XGBoost yg pakai squared error
    eval_metric='MAE',
    random_seed=42,
    early_stopping_rounds=30,
    verbose=50
)

cat_model.fit(
    X_train_cb, y_train,
    eval_set=(X_test_cb, y_test),
    use_best_model=True
)

# ============================================================
# SAVE CATBOOST MODEL
# ============================================================
cat_model.save_model(
    MODEL_DIR / "catboost_model.cbm"
)

print(f"CatBoost model saved to: {MODEL_DIR / 'catboost_model.cbm'}")

print("✓ CatBoost model saved.")

cat_pred_log = cat_model.predict(X_test_cb)
cat_pred_sec = np.expm1(cat_pred_log)

cat_mae = mean_absolute_error(
    test_xgb[TARGET],
    cat_pred_sec
)

print(f"CatBoost MAE : {cat_mae:.2f} detik")

cat_feature_importance = (
    pd.Series(
        cat_model.get_feature_importance(),
        index=FEATURES_RAW
    )
    .sort_values(ascending=False)
)

print("\nTop 10 CatBoost Feature Importance")
print(cat_feature_importance.head(10))

0:	learn: 0.5848876	test: 0.5728752	best: 0.5728752 (0)	total: 780ms	remaining: 6m 29s
50:	learn: 0.1868714	test: 0.1865589	best: 0.1865589 (50)	total: 22.5s	remaining: 3m 18s
100:	learn: 0.1604576	test: 0.1645884	best: 0.1645884 (100)	total: 41.4s	remaining: 2m 43s
150:	learn: 0.1496604	test: 0.1517320	best: 0.1517320 (150)	total: 59.5s	remaining: 2m 17s
200:	learn: 0.1463808	test: 0.1488047	best: 0.1488047 (200)	total: 1m 18s	remaining: 1m 57s
250:	learn: 0.1440000	test: 0.1466865	best: 0.1466865 (250)	total: 1m 38s	remaining: 1m 37s
300:	learn: 0.1417100	test: 0.1443221	best: 0.1443221 (300)	total: 1m 56s	remaining: 1m 17s
350:	learn: 0.1403079	test: 0.1430474	best: 0.1430474 (350)	total: 2m 14s	remaining: 57.3s
400:	learn: 0.1391769	test: 0.1419833	best: 0.1419833 (400)	total: 2m 33s	remaining: 37.8s
450:	learn: 0.1383401	test: 0.1412306	best: 0.1412278 (449)	total: 2m 53s	remaining: 18.9s
499:	learn: 0.1378492	test: 0.1409542	best: 0.1409542 (499)	total: 3m 11s	remaining: 0us

bes

In [33]:
train_df['prev_traveling_time_sec'] = train_df['prev_traveling_time_sec'].fillna(train_df['average_time_sec'])
test_df['prev_traveling_time_sec'] = test_df['prev_traveling_time_sec'].fillna(test_df['average_time_sec'])

for col in ['average_time_sec', 'prev_traveling_time_sec', 'segment_time_estimate']:
    train_df[col] = np.log1p(train_df[col])
    test_df[col] = np.log1p(test_df[col])

print('Sisa NaN:', train_df[SEQ_FEATURES].isna().sum().sum())

Sisa NaN: 0


In [34]:
# ============================================================
# 5.5 — PERSIAPAN SEQUENCE UNTUK LSTM
# ============================================================
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences

SEQ_FEATURES = [
    'stop_sequence','hour','day_of_week','trip_progress',
    'prev_deviation_ratio','prev_traveling_time_sec','cum_avg_deviation_in_trip',
    'average_time_sec','segment_time_estimate','bus_avg_deviation',
    'is_weekend','is_peak_hour','is_first_segment','is_gap_suspected'
]
N_SEGMENT_CLASSES = df_feat['segment_id_enc'].nunique()
MAX_LEN = int(df_feat.groupby('real_trip_id').size().max())

scaler = StandardScaler()
scaler.fit(train_df[SEQ_FEATURES].astype(float))

def build_sequences(d):
    d = d.sort_values(['real_trip_id','stop_sequence'])
    grouped = d.groupby('real_trip_id')

    X_num, X_seg, y_seq, w_seq = [], [], [], []
    for _, g in grouped:
        feats = scaler.transform(g[SEQ_FEATURES].astype(float))
        seg_ids = g['segment_id_enc'].values
        targets = np.log1p(g['traveling_time_sec'].values)
        weights = (~g['exclude_from_target'].values).astype(float)

        X_num.append(feats)
        X_seg.append(seg_ids)
        y_seq.append(targets)
        w_seq.append(weights)

    X_num = pad_sequences(X_num, maxlen=MAX_LEN, dtype='float32', padding='post')
    X_seg = pad_sequences(X_seg, maxlen=MAX_LEN, dtype='int32', padding='post')
    y_seq = pad_sequences(y_seq, maxlen=MAX_LEN, dtype='float32', padding='post')
    w_seq = pad_sequences(w_seq, maxlen=MAX_LEN, dtype='float32', padding='post')
    return X_num, X_seg, y_seq, w_seq

X_num_train, X_seg_train, y_train_seq, w_train_seq = build_sequences(train_df)
X_num_test, X_seg_test, y_test_seq, w_test_seq = build_sequences(test_df)

print('Shape X_num_train:', X_num_train.shape, '| MAX_LEN:', MAX_LEN)

Shape X_num_train: (14675, 35, 14) | MAX_LEN: 35


In [ ]:
# ============================================================
# 5.6 — MEMBANGUN & TRAINING LSTM
# ============================================================
from tensorflow.keras.layers import Input, Embedding, Concatenate, LSTM, TimeDistributed, Dense, Masking
from tensorflow.keras.models import Model

num_input = Input(shape=(MAX_LEN, len(SEQ_FEATURES)), name='numeric_features')
seg_input = Input(shape=(MAX_LEN,), name='segment_id')

seg_embed = Embedding(input_dim=N_SEGMENT_CLASSES + 1, output_dim=8, mask_zero=False)(seg_input)
merged = Concatenate()([num_input, seg_embed])
masked = Masking(mask_value=0.0)(merged)

x = LSTM(64, return_sequences=True)(masked)
x = LSTM(32, return_sequences=True)(x)
output = TimeDistributed(Dense(1))(x)

lstm_model = Model(inputs=[num_input, seg_input], outputs=output)
lstm_model.compile(optimizer=tf.keras.optimizers.Adam(clipnorm=1.0), loss='mae')  # <-- baris yang diubah
lstm_model.summary()

history = lstm_model.fit(
    [X_num_train, X_seg_train], y_train_seq[..., np.newaxis],
    sample_weight=w_train_seq,
    validation_data=([X_num_test, X_seg_test], y_test_seq[..., np.newaxis], w_test_seq),
    epochs=30, batch_size=64, verbose=1
)

# ============================================================
# SAVE LSTM MODEL
# ============================================================
lstm_model.save(
    MODEL_DIR / "lstm_model.keras"
)

print("✓ LSTM model saved.")

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ segment_id          │ (None, 35)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ numeric_features    │ (None, 35, 14)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, 35, 8)     │        360 │ segment_id[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_2       │ (None, 35, 22)    │          0 │ numeric_features… │
│ (Concatenate)       │                   │            │ embedding_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_2         │ (None, 35, 22)    │          0 │ concatenate_2[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ masking_2 (Masking) │ (None, 35, 22)    │          0 │ concatenate_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ any_2 (Any)         │ (None, 35)        │          0 │ not_equal_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_4 (LSTM)       │ (None, 35, 64)    │     22,272 │ masking_2[0][0],  │
│                     │                   │            │ any_2[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_5 (LSTM)       │ (None, 35, 32)    │     12,416 │ lstm_4[0][0],     │
│                     │                   │            │ any_2[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_2  │ (None, 35, 1)     │         33 │ lstm_5[0][0],     │
│ (TimeDistributed)   │                   │            │ any_2[0][0]       │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 35,081 (137.04 KB)

 Trainable params: 35,081 (137.04 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
230/230 ━━━━━━━━━━━━━━━━━━━━ 22s 68ms/step - loss: 0.4604 - val_loss: 0.1213
Epoch 2/30
230/230 ━━━━━━━━━━━━━━━━━━━━ 20s 65ms/step - loss: 0.1043 - val_loss: 0.0942
Epoch 3/30
230/230 ━━━━━━━━━━━━━━━━━━━━ 15s 65ms/step - loss: 0.0937 - val_loss: 0.0911
Epoch 4/30
230/230 ━━━━━━━━━━━━━━━━━━━━ 14s 63ms/step - loss: 0.0904 - val_loss: 0.0882
Epoch 5/30
230/230 ━━━━━━━━━━━━━━━━━━━━ 14s 63ms/step - loss: 0.0886 - val_loss: 0.0881
Epoch 6/30
230/230 ━━━━━━━━━━━━━━━━━━━━ 15s 64ms/step - loss: 0.0878 - val_loss: 0.0875
Epoch 7/30
230/230 ━━━━━━━━━━━━━━━━━━━━ 14s 63ms/step - loss: 0.0872 - val_loss: 0.0870
Epoch 8/30
230/230 ━━━━━━━━━━━━━━━━━━━━ 14s 63ms/step - loss: 0.0867 - val_loss: 0.0876
Epoch 9/30
230/230 ━━━━━━━━━━━━━━━━━━━━ 15s 65ms/step - loss: 0.0862 - val_loss: 0.0865
Epoch 10/30
230/230 ━━━━━━━━━━━━━━━━━━━━ 20s 62ms/step - loss: 0.0859 - val_loss: 0.0855
Epoch 11/30
230/230 ━━━━━━━━━━━━━━━━━━━━ 15s 64ms/step - loss: 0.0856 - val_loss: 0.0872
Epoch 12/30
230/230 ━━━━━━━━━━

In [ ]:
# ============================================================
# 5.7 — PREDIKSI LSTM (UNPAD) & ENSEMBLE (XGB + CATBOOST + LSTM)
# ============================================================
lstm_pred_log_padded = lstm_model.predict([X_num_test, X_seg_test]).squeeze(-1)

test_sorted = test_df.sort_values(['real_trip_id','stop_sequence'])
trip_lengths = test_sorted.groupby('real_trip_id').size().values

lstm_pred_log_flat = []
for i, length in enumerate(trip_lengths):
    lstm_pred_log_flat.extend(lstm_pred_log_padded[i, :length])
test_sorted = test_sorted.copy()
test_sorted['lstm_pred_log'] = lstm_pred_log_flat
test_sorted['lstm_pred_sec'] = np.expm1(test_sorted['lstm_pred_log'])

eval_df = test_sorted[~test_sorted['exclude_from_target']].copy()
eval_df = eval_df.merge(
    test_xgb[['real_trip_id','stop_sequence']].assign(
        xgb_pred_sec=xgb_pred_sec,
        cat_pred_sec=cat_pred_sec
    ),
    on=['real_trip_id','stop_sequence'], how='inner'
)

WEIGHT_XGB = 0.5
eval_df['ensemble_xgb_lstm'] = WEIGHT_XGB*eval_df['xgb_pred_sec'] + (1-WEIGHT_XGB)*eval_df['lstm_pred_sec']
eval_df['ensemble_cat_lstm'] = WEIGHT_XGB*eval_df['cat_pred_sec'] + (1-WEIGHT_XGB)*eval_df['lstm_pred_sec']
eval_df['ensemble_xgb_cat_lstm'] = (1/3)*(eval_df['xgb_pred_sec'] + eval_df['cat_pred_sec'] + eval_df['lstm_pred_sec'])

for col in ['xgb_pred_sec','cat_pred_sec','lstm_pred_sec',
            'ensemble_xgb_lstm','ensemble_cat_lstm','ensemble_xgb_cat_lstm']:
    mae = mean_absolute_error(eval_df['traveling_time_sec'], eval_df[col])
    print(f'{col:25s} MAE: {mae:.2f} detik')

prediction_path = OUTPUT_DIR / "test_predictions.parquet"

eval_df.to_parquet(
    prediction_path,
    index=False
)

print(f"\nPrediction saved to: {prediction_path}")

115/115 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step
xgb_pred_sec              MAE: 105.35 detik
cat_pred_sec              MAE: 110.23 detik
lstm_pred_sec             MAE: 108.66 detik
ensemble_xgb_lstm         MAE: 99.88 detik
ensemble_cat_lstm         MAE: 103.61 detik
ensemble_xgb_cat_lstm     MAE: 99.88 detik

Saved to test_predictions.parquet


## Fase 6 — Evaluasi Model

Mengukur performa model dengan 4 metrik sesuai permintaan soal: **MAE**, **RMSE**, **MAPE per segmen**, dan **MAE per putaran** (agregat seluruh trip, bukan per-baris). Turut disertakan **baseline naive** (`average_time_sec` sebagai prediksi langsung) sebagai pembanding — untuk menunjukkan baseline mana yang harus dikalahkan model agar bernilai secara bisnis, dan kapan ensemble tidak memberi keuntungan signifikan.

In [ ]:
# ============================================================
# 6.1 — MAE & RMSE OVERALL (TERMASUK BASELINE NAIVE)
# ============================================================
from sklearn.metrics import mean_absolute_error, mean_squared_error

eval_df['naive_pred_sec'] = eval_df['average_time_sec']

model_cols = ['naive_pred_sec', 'xgb_pred_sec', 'cat_pred_sec', 'lstm_pred_sec',
              'ensemble_xgb_lstm', 'ensemble_cat_lstm', 'ensemble_xgb_cat_lstm']

results = []
for col in model_cols:
    mae = mean_absolute_error(eval_df['traveling_time_sec'], eval_df[col])
    mse = mean_squared_error(eval_df['traveling_time_sec'], eval_df[col])
    rmse = np.sqrt(mse)
    results.append({'model': col, 'MAE': mae, 'RMSE': rmse})

results_df = pd.DataFrame(results).sort_values('MAE')
print(results_df.to_string(index=False))

results_df.to_csv(OUTPUT_DIR / "model_comparison_overall.csv", index=False)

                model        MAE        RMSE
ensemble_xgb_cat_lstm  99.877256 1041.379828
    ensemble_xgb_lstm  99.881309 1027.932474
    ensemble_cat_lstm 103.608557 1080.288325
         xgb_pred_sec 105.351023 1206.411373
        lstm_pred_sec 108.663812 1133.039812
         cat_pred_sec 110.233117 1326.271979
       naive_pred_sec 312.216563 2266.459006


In [ ]:
# ============================================================
# 6.2 — MAPE PER SEGMEN
# ============================================================
import numpy as np

def mape(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    # safety: hindari division by zero
    eps = 1e-8
    return np.mean(np.abs((y_true - y_pred) / (y_true + eps))) * 100


BEST_MODEL_COL = 'ensemble_xgb_lstm'  # model terbaik dari 6.1

mape_per_segment = eval_df.groupby('segment_id').apply(
    lambda g: mape(g['traveling_time_sec'], g[BEST_MODEL_COL])
).reset_index(name='mape')

mape_per_segment = mape_per_segment.merge(
    eval_df.groupby('segment_id').size().reset_index(name='n_samples'),
    on='segment_id'
)

print('MAPE rata-rata seluruh segmen:', mape_per_segment['mape'].mean())
print('\nTop 5 segmen dengan MAPE terburuk:')
print(mape_per_segment.sort_values('mape', ascending=False).head(5))

print('\nTop 5 segmen dengan MAPE terbaik:')
print(mape_per_segment.sort_values('mape').head(5))

mape_per_segment.to_csv(
    OUTPUT_DIR / "mape_per_segment.csv",
    index=False
)

MAPE rata-rata seluruh segmen: 45.06673997075835

Top 5 segmen dengan MAPE terburuk:
          segment_id        mape  n_samples
15  SEG-DUMMY-008940  751.585040       1566
2   SEG-DUMMY-008734  655.017139       1774
34  SEG-DUMMY-009393   39.784417        110
20  SEG-DUMMY-009042   37.601999        420
3   SEG-DUMMY-008745   31.431582         78

Top 5 segmen dengan MAPE terbaik:
          segment_id      mape  n_samples
26  SEG-DUMMY-009168  6.388371       2071
10  SEG-DUMMY-008829  6.758263       1804
24  SEG-DUMMY-009049  6.778841       1538
40  SEG-DUMMY-009420  7.735528       1504
17  SEG-DUMMY-008954  8.059649       2027


/tmp/ipykernel_2244/879277503.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mape_per_segment = eval_df.groupby('segment_id').apply(


In [ ]:
# ============================================================
# 6.3 — MAE PER PUTARAN (LEVEL TRIP, BUKAN PER-BARIS)
# ============================================================
import numpy as np

trip_agg = eval_df.groupby('real_trip_id').agg(
    actual_total_sec=('traveling_time_sec', 'sum'),
    pred_total_sec=(BEST_MODEL_COL, 'sum'),
    n_segments=('traveling_time_sec', 'count')
).reset_index()

# error absolut per trip
trip_agg['abs_error_sec'] = (trip_agg['actual_total_sec'] - trip_agg['pred_total_sec']).abs()

# MAE level trip
mae_putaran = trip_agg['abs_error_sec'].mean()

print(f'MAE per putaran (akumulasi total waktu trip): {mae_putaran:.2f} detik (~{mae_putaran/60:.1f} menit)')
print('\nDistribusi error per putaran:')
print(trip_agg['abs_error_sec'].describe())

trip_agg.to_csv(
    OUTPUT_DIR / "mae_per_trip.csv",
    index=False
)

MAE per putaran (akumulasi total waktu trip): 1670.71 detik (~27.8 menit)

Distribusi error per putaran:
count     3624.000000
mean      1670.705929
std       4166.202099
min          0.035797
25%         80.196052
50%        209.945924
75%        763.864562
max      34919.101031
Name: abs_error_sec, dtype: float64


In [40]:
# ============================================================
# 6.4 — RINGKASAN FINAL UNTUK WRITE-UP
# ============================================================
print('=== RINGKASAN EVALUASI ===')
print(f'Model terbaik: {BEST_MODEL_COL}')
print(f'MAE overall: {results_df[results_df.model==BEST_MODEL_COL].MAE.values[0]:.2f} detik')
print(f'RMSE overall: {results_df[results_df.model==BEST_MODEL_COL].RMSE.values[0]:.2f} detik')
print(f'MAPE rata-rata per segmen: {mape_per_segment.mape.mean():.2f}%')
print(f'MAE per putaran: {mae_putaran:.2f} detik')
print(f'\nPerbaikan vs baseline naive (average_time_sec):')
naive_mae = results_df[results_df.model=='naive_pred_sec'].MAE.values[0]
improvement = (naive_mae - results_df[results_df.model==BEST_MODEL_COL].MAE.values[0]) / naive_mae * 100
print(f'{improvement:.1f}% lebih baik dari baseline naive')

=== RINGKASAN EVALUASI ===
Model terbaik: ensemble_xgb_lstm
MAE overall: 99.88 detik
RMSE overall: 1027.93 detik
MAPE rata-rata per segmen: 45.07%
MAE per putaran: 1670.71 detik

Perbaikan vs baseline naive (average_time_sec):
68.0% lebih baik dari baseline naive
